### Pós-graduação em Ciência de Dados e Machine Learning

#### Disciplina: **Introdução a Redes Neurais**

#### Projeto Final para disciplina Introdução a Redes Neurais

<BR>
    

#### Nome dos integrantes:  JOÃO PEDRO ANTONIO DOS SANTOS CASTRO | MATHEUS MENDES NEVES
#### RA: 52400204
#### RA: 22000917


## 🧠 Justificativa Técnica com Base em Artigo Científico

### 📄 Artigo Utilizado
**Título**: *Attributes as Operators: Factorizing Unseen Attribute-Object Compositions*  
**Autores**: Tushar Nagarajan & Kristen Grauman  
**Instituições**: University of Texas at Austin, Facebook AI Research  
**Link**: [https://arxiv.org/abs/1803.09851](https://arxiv.org/abs/1803.09851)

---

### 🔍 Resumo do Artigo

O artigo propõe uma abordagem alternativa para o aprendizado de atributos visuais. Em vez de tratar atributos como vetores fixos (como se fossem objetos), os autores modelam **atributos como operadores** — ou seja, transformações que modificam a representação dos objetos. Isso permite reconhecer composições **inéditas de atributos e objetos**, mesmo que não tenham sido vistas durante o treinamento.

Além disso, os autores utilizam **regularizadores linguísticos** para reforçar o aprendizado:
- **Comutatividade**: a ordem dos atributos não altera o resultado (ex: *banana madura fatiada* = *banana fatiada madura*);
- **Antônimos**: certos atributos devem anular outros (ex: *cego* anula *afiado*);
- **Consistência inversa**: ao trocar um atributo por outro, o modelo deve produzir mudanças coerentes.

---

### 🏥 Aplicação ao Nosso Projeto

Neste trabalho, aplicamos deep learning para **detecção de pneumonia em imagens de raio-X**. Apesar de não tratarmos atributos linguísticos diretamente, adotamos o **princípio central do artigo**: usar redes convolucionais que consigam aprender transformações complexas nas imagens.

---

### 🤖 Arquitetura Utilizada: **DenseNet121**

Baseamo-nos também em evidências da literatura médica (ex: *CheXNet*, Rajpurkar et al., 2017), que demonstram a eficácia da DenseNet121 para diagnóstico em imagens de tórax.

**Motivos da escolha da DenseNet121**:
- Arquitetura densa que **melhora o fluxo de gradientes** entre camadas;
- Promove **reaproveitamento de recursos computacionais** com conexões diretas entre camadas;
- Excelente desempenho em tarefas com imagens médicas e conjuntos de dados limitados;
- Pré-treinada no ImageNet, permitindo **transfer learning**, acelerando o treinamento com boa generalização.

---

### ✅ Conclusão

Nosso projeto adota o uso de redes convolucionais (CNNs) avançadas, alinhado com os princípios do artigo citado, visando uma solução robusta para um problema real. A arquitetura escolhida foi **justificada com base em artigos científicos**, atendendo aos critérios exigidos para o trabalho final.


# Classificação de Pneumonia em Raios-X Torácicos

**Objetivo**: Desenvolver um modelo de Deep Learning para classificar imagens de raios-X torácicos em "Normal" ou "Pneumonia".

**Dataset**: Chest X-Ray Images (Pneumonia) do Kaggle, com a seguinte estrutura:
- `chest_xray/train/`
  - `NORMAL/`
  - `PNEUMONIA/`
- `chest_xray/test/`
  - `NORMAL/`
  - `PNEUMONIA/`
- `chest_xray/val/`
  - `NORMAL/`
  - `PNEUMONIA/`

**Base Teórica**: Inspirado no artigo CheXNet (Rajpurkar et al., 2017) que utilizou DenseNet-121 para alcançar desempenho de nível radiologista.


In [ ]:
# Importação de bibliotecas
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import cv2

import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

In [ ]:
import tensorflow as tf
print("GPU disponível:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Verificando a estrutura dos diretórios
base_dir = 'chest_xray/chest_xray'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

print("Diretório de treino:", os.listdir(train_dir))
print("Diretório de validação:", os.listdir(val_dir))
print("Diretório de teste:", os.listdir(test_dir))

In [ ]:
# Análise exploratória - Contagem de imagens em cada conjunto
def count_images(directory):
    normal = len(os.listdir(os.path.join(directory, 'NORMAL')))
    pneumonia = len(os.listdir(os.path.join(directory, 'PNEUMONIA')))
    return normal, pneumonia

train_normal, train_pneumonia = count_images(train_dir)
val_normal, val_pneumonia = count_images(val_dir)
test_normal, test_pneumonia = count_images(test_dir)

print(f"Treino - Normal: {train_normal}, Pneumonia: {train_pneumonia}")
print(f"Validação - Normal: {val_normal}, Pneumonia: {val_pneumonia}")
print(f"Teste - Normal: {test_normal}, Pneumonia: {test_pneumonia}")

In [ ]:
# Criando um gráfico de barras
labels = ['Treino', 'Validação', 'Teste']
normal_counts = [train_normal, val_normal, test_normal]
pneumonia_counts = [train_pneumonia, val_pneumonia, test_pneumonia]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, normal_counts, width, label='Normal')
rects2 = ax.bar(x + width/2, pneumonia_counts, width, label='Pneumonia')

ax.set_ylabel('Número de Imagens')
ax.set_title('Distribuição de Imagens por Classe')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# Pré-processamento e aumento de dados
img_size = 224  # antes: 224
batch_size = 32

In [ ]:
# Data augmentation para o conjunto de treino
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [ ]:
# Para validação e teste, apenas normalização
val_test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
# Geradores de dados
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False # ANtes True
)


val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)


In [ ]:
# Visualização de algumas imagens do dataset
def plot_images(images_arr):
    fig, axes = plt.subplots(1, 5, figsize=(20, 20))
    axes = axes.flatten()
    for img, ax in zip(images_arr, axes):
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

sample_images, _ = next(train_generator)
plot_images(sample_images[:5])

In [ ]:
# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-6
)

In [ ]:
def build_model():
    base_model = DenseNet121(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling=None
    )
    base_model.trainable = False


    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs, outputs)
    print("Camadas treináveis:", len([l for l in model.layers if l.trainable]))

    
    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC()]
    )
    return model

# 3. Construir o modelo
model = build_model()

# 4. Verificar a arquitetura
model.summary()

In [ ]:
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=[early_stopping, reduce_lr]
)


In [ ]:
# Visualização do histórico de treinamento
def plot_training_history(history):
    plt.figure(figsize=(12, 4))
    
    # Gráfico de acurácia
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Treino')
    plt.plot(history.history['val_accuracy'], label='Validação')
    plt.title('Acurácia por Época')
    plt.xlabel('Época')
    plt.ylabel('Acurácia')
    plt.legend()
    
    # Gráfico de loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Treino')
    plt.plot(history.history['val_loss'], label='Validação')
    plt.title('Loss por Época')
    plt.xlabel('Época')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

plot_training_history(history)

In [ ]:
# Avaliação no conjunto de teste
test_loss, test_acc, test_auc = model.evaluate(test_generator)
print(f'\nAcurácia no teste: {test_acc:.2%}')
print(f'AUC no teste: {test_auc:.2%}')

In [ ]:
# Matriz de confusão e relatório de classificação
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Previsões no conjunto de teste
test_generator.reset()
y_pred = model.predict(test_generator)
y_pred = (y_pred > 0.5).astype(int)
y_true = test_generator.classes

# Matriz de confusão
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Pneumonia'], 
            yticklabels=['Normal', 'Pneumonia'])
plt.ylabel('Verdadeiro')
plt.xlabel('Predito')
plt.title('Matriz de Confusão')
plt.show()

# Relatório de classificação
print(classification_report(y_true, y_pred, target_names=['Normal', 'Pneumonia']))

In [ ]:
model.save('modelo_pneumonia.h5')
print("Modelo salvo como 'modelo_pneumonia.h5'")

## 🧪 Teste de Inferência com Nova Imagem

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np


img_path = "pneumonia.jpg"  # <-- Substitua por sua imagem
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
class_index = np.argmax(prediction)
confidence = np.max(prediction)

print(f"Classe prevista: {class_index} (Confiança: {confidence:.2f})")

In [ ]:
# Carregando o modelo salvo
loaded_model = tf.keras.models.load_model('modelo_pneumonia.h5')
# Verificando a arquitetura do modelo carregado
loaded_model.summary()
# Avaliando o modelo carregado no conjunto de teste
test_loss, test_acc, test_auc = loaded_model.evaluate(test_generator)
print(f'\nAcurácia no teste: {test_acc:.2%}')
print(f'AUC no teste: {test_auc:.2%}')
# Previsões no conjunto de teste com o modelo carregado
loaded_model.predict(test_generator)
# Visualizando algumas previsões
def plot_predictions(images, predictions, true_labels):
    fig, axes = plt.subplots(1, 5, figsize=(20, 20))
    axes = axes.flatten()
    for img, pred, true_label, ax in zip(images, predictions, true_labels, axes):
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'Predito: {pred[0]:.2f}, Verdadeiro: {true_label}')
    plt.tight_layout()
    plt.show()
sample_images, _ = next(test_generator)
sample_images = sample_images[:5]
sample_predictions = loaded_model.predict(sample_images)
sample_predictions = (sample_predictions > 0.5).astype(int)
sample_true_labels = test_generator.classes[:5]
plot_predictions(sample_images, sample_predictions, sample_true_labels)

